In [2]:
!pip install osmnx networkx folium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.9 MB/s eta 0:00:00


Cell 1: Imports and Network Download

In [3]:
import osmnx as ox
import networkx as nx

# Configure OSMnx to use cache (speeds up subsequent runs)
ox.settings.use_cache = True

# Download the drivable street network for Manhattan
place = "Manhattan, New York City, New York, USA"
G = ox.graph_from_place(place, network_type="drive")

# Impute missing speed limits and calculate travel time for each road segment
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)

print(f"Network downloaded: {len(G.nodes)} nodes and {len(G.edges)} edges.")

Network downloaded: 4635 nodes and 9916 edges.


Cell 2: Define Origin/Destination and Calculate Route

In [5]:
# Define coordinates (Latitude, Longitude)
# Origin: Times Square
orig_lat, orig_lon = 40.7580, -73.9855
# Destination: Washington Square Park
dest_lat, dest_lon = 40.7308, -73.9973

# Find the nearest network nodes to these coordinates
orig_node = ox.distance.nearest_nodes(G, X=orig_lon, Y=orig_lat)
dest_node = ox.distance.nearest_nodes(G, X=dest_lon, Y=dest_lat)

# Calculate the shortest path based on 'travel_time'
route = nx.shortest_path(G, orig_node, dest_node, weight="travel_time")

# Retrieve travel times for the route edges using OSMnx 2.x compatible routing functions
route_edges = ox.routing.route_to_gdf(G, route)
travel_time_sec = route_edges["travel_time"].sum()

print(f"Estimated Driving Time: {travel_time_sec / 60:.1f} minutes")

Estimated Driving Time: 5.2 minutes


Cell 3: Generate the Interactive Map

In [9]:
# Plot the route using the CartoDB positron tile provider to avoid blocks
route_map = route_edges.explore(
    color="blue",
    style_kwds={"weight": 5},
    tiles="Esri.WorldGrayCanvas"
)

# Calling the variable at the end of the cell displays it in the notebook output
route_map